In [ ]:
# app.py
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
import uuid

st.set_page_config(page_title="Space Cleaner - Prototype", layout="wide")

# ---------- Session state init ----------
if "debris" not in st.session_state:
    st.session_state.debris = []   # list of dicts: {'id','x','y','size'}
if "satellites" not in st.session_state:
    st.session_state.satellites = []  # list of dicts: {'id','x','y'}
if "score" not in st.session_state:
    st.session_state.score = 0

# ---------- Helpers ----------
def spawn_debris(n=1):
    for _ in range(n):
        angle = np.random.uniform(0, 2 * np.pi)
        r = np.random.uniform(0.6, 1.1)  # orbit radius (normalized)
        x, y = r * np.cos(angle), r * np.sin(angle)
        st.session_state.debris.append({
            "id": str(uuid.uuid4()),
            "x": float(x),
            "y": float(y),
            "size": float(np.random.uniform(20, 80))
        })

def launch_vessel(thrust: float, quality: float):
    # thrust, quality in [0,100]
    # success probability formula simple and explainable
    success_prob = 0.2 + 0.7 * (quality / 100.0) * (thrust / 100.0)
    success_prob = min(0.95, max(0.0, success_prob))
    success = np.random.rand() < success_prob
    if success:
        # spawn one satellite at random orbit position
        angle = np.random.uniform(0, 2 * np.pi)
        r = np.random.uniform(0.7, 1.0)
        x, y = r * np.cos(angle), r * np.sin(angle)
        st.session_state.satellites.append({"id": str(uuid.uuid4()), "x": float(x), "y": float(y)})
        return True, success_prob
    else:
        # spawn 1-4 debris pieces
        n = np.random.randint(1, 5)
        spawn_debris(n)
        return False, success_prob

def collect_debris(cleaner_x, cleaner_y, radius):
    removed = []
    for d in list(st.session_state.debris):
        dist = np.hypot(d["x"] - cleaner_x, d["y"] - cleaner_y)
        if dist <= radius:
            removed.append(d)
            st.session_state.debris.remove(d)
            st.session_state.score += 10
    return removed

# ---------- Layout ----------
st.title("🚀 Space Cleaner — Prototype")
left, right = st.columns([3, 1])

with right:
    st.header("Controls")
    thrust = st.slider("Thrust", 0, 100, 70)
    quality = st.slider("Build quality", 0, 100, 60)
    if st.button("Launch"):
        ok, p = launch_vessel(thrust, quality)
        if ok:
            st.success(f"Launch success! (p={p:.2f}) — satellite deployed.")
        else:
            st.warning(f"Launch failed → debris created (p_success={p:.2f}).")

    st.markdown("---")
    st.subheader("Cleaner controls")
    cleaner_x = st.slider("Cleaner X", -1.5, 1.5, 0.0, step=0.05)
    cleaner_y = st.slider("Cleaner Y", -1.5, 1.5, 0.0, step=0.05)
    collect_radius = st.slider("Collect radius", 0.05, 0.6, 0.2, step=0.01)
    if st.button("Collect"):
        removed = collect_debris(cleaner_x, cleaner_y, collect_radius)
        st.info(f"Collected {len(removed)} debris. Score = {st.session_state.score}")

    if st.button("Reset Game State"):
        st.session_state.debris = []
        st.session_state.satellites = []
        st.session_state.score = 0
        st.success("Game state reset.")

with left:
    # Draw map
    fig, ax = plt.subplots(figsize=(6, 6))
    # Earth as circle
    earth = plt.Circle((0, 0), 0.35, color="deepskyblue", alpha=0.6)
    ax.add_artist(earth)

    # plot satellites
    if st.session_state.satellites:
        xs = [s["x"] for s in st.session_state.satellites]
        ys = [s["y"] for s in st.session_state.satellites]
        ax.scatter(xs, ys, marker="*", s=180, label="satellite", zorder=3)

    # plot debris
    if st.session_state.debris:
        xs = [d["x"] for d in st.session_state.debris]
        ys = [d["y"] for d in st.session_state.debris]
        sizes = [d["size"] for d in st.session_state.debris]
        ax.scatter(xs, ys, c="red", s=sizes, alpha=0.7, label="debris", zorder=2)

    # plot cleaner
    ax.scatter(cleaner_x, cleaner_y, c="limegreen", s=140, marker="s", label="cleaner", zorder=4)

    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")
    st.pyplot(fig)

    st.markdown(f"**Score:** {st.session_state.score}  —  **Debris count:** {len(st.session_state.debris)}  —  **Satellites:** {len(st.session_state.satellites)}")


In [5]:
pip install streamlit pillow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import streamlit as st
from PIL import Image, ImageDraw
import random
import streamlit.components.v1 as components

In [2]:
pip install streamlit


  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/10.1 MB 8.4 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/10.1 MB 11.5 MB/s eta 0:00:01
   -------------------------------- ------- 8.1/10.1 MB 15.7 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.1 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 12.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/731.2 kB ? eta -:--:--
   --------------------------------------- 731.2/731.2 kB 14.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   --------------------------------- ------ 5.8/6.9 MB 29.4 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 15.7 MB/s eta 0:00:00
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ---------------------------------------- 0.0/26

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install pygame

Note: you may need to restart the kernel to use updated packages.Collecting pygame
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.6 MB 8.3 MB/s eta 0:00:02
   ----------------- ---------------------- 4.7/10.6 MB 15.0 MB/s eta 0:00:01
   ------------------------------------- -- 10.0/10.6 MB 19.4 MB/s eta 0:00:01
   ---------------------------------------  10.5/10.6 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------  10.5/10.6 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 11.0 MB/s eta 0:00:00




[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
